# Satlas Super-Resolution Inference

This notebook runs a trained **Satlas ESRGAN** model on a directory of
low-resolution (LR) tiles and saves the super-resolved (SR) outputs as
PNG files. The resulting tiles can then be stitched into a full map
using the `Tile_Stitching_Mapping.ipynb` notebook.

## What this notebook does

- Loads a trained RRDBNet (ESRGAN) checkpoint from the Satlas repo.
- Iterates over all LR tiles in the input directory.
- Runs 4x super-resolution per tile.
- Saves results as `<tile_id>.png` in the output directory.

## What this notebook does *not* do

- **SwinIR inference** — will be added later in its own notebook / section.
- **Stitching** — see `Tile_Stitching_Mapping.ipynb`.
- **Evaluation (PSNR / CLIP)** — see `SSR_Evaluation.ipynb`.

## Prerequisites

1. The [satlas-super-resolution](https://github.com/allenai/satlas-super-resolution)
   repository must be cloned locally and importable (`basicsr` package available).
2. A trained checkpoint (`net_g_latest.pth` or similar) must exist.
3. LR input tiles must be in a single directory.

## Two inference options

1. **Python (recommended for notebooks)** — direct model load + loop. Shown below.
2. **Satlas CLI** — run the official `run_ssr_infer.py` script with a YAML config.
   Useful if you want to reuse Satlas' built-in pre/post-processing or sweep
   multiple model configs. Shown at the bottom.

In [ ]:
# Imports
from pathlib import Path
import cv2
import numpy as np
import torch
from tqdm import tqdm

from basicsr.archs.rrdbnet_arch import RRDBNet

In [ ]:
# Configuration
# Adjust these paths to your local setup.

# Path to the cloned satlas-super-resolution repo (only needed for the CLI option)
SATLAS_REPO = Path('satlas-super-resolution')

# Trained checkpoint
MODEL_PATH = SATLAS_REPO / 'experiments' / 'esrgan_best_optuna' / 'models' / 'net_g_latest.pth'

# LR input tiles (e.g. 128x128 jpgs)
INPUT_DIR = Path('data/inputs')
INPUT_GLOB = '*.jpg'

# Where to write SR output tiles (consumed by mapping.ipynb as SATLAS_TILES_DIR)
OUTPUT_DIR = Path('data/satlas_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model architecture settings (must match the trained checkpoint)
SCALE = 4
NUM_FEAT = 64
NUM_BLOCK = 23
NUM_GROW_CH = 32

# Skip tiles that already have an output (useful for resuming long runs)
SKIP_EXISTING = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Input : {INPUT_DIR}')
print(f'Output: {OUTPUT_DIR}')

## Load the model

BasicSR checkpoints can be stored in three common formats:
- a plain `state_dict`,
- a dict with a `params` key (EMA-free weights),
- a dict with a `params_ema` key (EMA weights, usually preferred for inference).

The helper below picks the right one automatically and prefers EMA weights.

In [ ]:
def load_rrdbnet(
    checkpoint_path: Path,
    device: torch.device,
    scale: int = 4,
    num_feat: int = 64,
    num_block: int = 23,
    num_grow_ch: int = 32,
) -> RRDBNet:
    """Load an RRDBNet checkpoint, handling the common BasicSR key formats."""
    model = RRDBNet(
        num_in_ch=3,
        num_out_ch=3,
        num_feat=num_feat,
        num_block=num_block,
        num_grow_ch=num_grow_ch,
        scale=scale)

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and 'params_ema' in checkpoint:
        state = checkpoint['params_ema']
        source = 'params_ema'
    elif isinstance(checkpoint, dict) and 'params' in checkpoint:
        state = checkpoint['params']
        source = 'params'
    else:
        state = checkpoint
        source = 'state_dict'

    model.load_state_dict(state, strict=True)
    model.eval().to(device)
    print(f'Loaded checkpoint from {checkpoint_path.name} (key: {source})')
    return model


model = load_rrdbnet(
    MODEL_PATH,
    device=DEVICE,
    scale=SCALE,
    num_feat=NUM_FEAT,
    num_block=NUM_BLOCK,
    num_grow_ch=NUM_GROW_CH)

## Run inference

Each LR tile is read with OpenCV (BGR), converted to RGB, normalized to
`[0, 1]`, sent through the model, and written back as an 8-bit PNG.

In [ ]:
@torch.no_grad()
def super_resolve(img_bgr: np.ndarray, model: RRDBNet, device: torch.device) -> np.ndarray:
    """Run one SR forward pass. Input/output are HxWx3 uint8 BGR arrays."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(img_rgb).float().div(255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)

    sr = model(tensor).squeeze(0).clamp(0, 1).cpu()
    sr = (sr.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return cv2.cvtColor(sr, cv2.COLOR_RGB2BGR)


def run_inference(
    model: RRDBNet,
    input_dir: Path,
    output_dir: Path,
    device: torch.device,
    input_glob: str = '*.jpg',
    skip_existing: bool = True,
) -> None:
    """Run SR on every tile in `input_dir` and save PNGs to `output_dir`."""
    lr_files = sorted(input_dir.glob(input_glob))
    if not lr_files:
        raise RuntimeError(f'No images found in {input_dir} matching {input_glob}')

    output_dir.mkdir(parents=True, exist_ok=True)
    skipped, processed, failed = 0, 0, 0

    for lr_path in tqdm(lr_files, desc='SR inference'):
        save_path = output_dir / f'{lr_path.stem}.png'
        if skip_existing and save_path.exists():
            skipped += 1
            continue

        img = cv2.imread(str(lr_path), cv2.IMREAD_COLOR)
        if img is None:
            print(f'Could not load: {lr_path.name}')
            failed += 1
            continue

        sr = super_resolve(img, model, device)
        cv2.imwrite(str(save_path), sr)
        processed += 1

    print(f'\nDone. Processed: {processed} | Skipped: {skipped} | Failed: {failed}')
    print(f'Outputs in: {output_dir}')


run_inference(
    model=model,
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    input_glob=INPUT_GLOB,
    skip_existing=SKIP_EXISTING)

---

## Alternative: Satlas CLI

The Satlas repo ships its own inference entrypoint, driven by a YAML
config. This produces equivalent results to the Python loop above, but
uses Satlas' built-in pre/post-processing. Useful when comparing
multiple checkpoints or configs without editing notebook code.

The YAML should set:
- `network_g`: same `num_feat`/`num_block`/`num_grow_ch`/`scale` as the checkpoint,
- `path.pretrain_network_g`: path to the `.pth` file,
- `datasets.test.dataroot_lq`: path to LR tiles,
- output directory.

Run the cell below to invoke it. Adjust `INFER_SCRIPT` and `INFER_CONFIG`
if your paths differ.

In [ ]:
INFER_SCRIPT = SATLAS_REPO / 'run_ssr_infer.py'
INFER_CONFIG = SATLAS_REPO / 'ssr' / 'options' / 'best_esrgan_infer_no_gt.yml'

!python {INFER_SCRIPT} -opt {INFER_CONFIG}

---